> ## SOLUTION / ANSWER KEY &mdash; Lab 10.6
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-06-fairness-across-groups.ipynb`](../lab-06-fairness-across-groups.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 10.6 &mdash; Fairness Across Groups

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 5 &middot; Module 10 &mdash; Ethics &amp; Responsible AI in Agentic Systems**

### What you'll do
- Compute an outcome rate for each group
- Flag disparate impact with the 80% rule
- See why an average can hide unfairness

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it &amp; observe**. The responsible-AI logic here &mdash; injection defence, least privilege, trace-reading, fairness, the eval loop, the guardrails &mdash; is **real, deterministic Python** you can run offline. The **Advanced agent labs (10&ndash;12)** additionally drive a **real Groq model** through `create_agent`: **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a responsible agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The agent labs use a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`); the guardrail/eval/trace logic is genuine rule-based Python. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. This is the **course finale** &mdash; Lab 5.2: responsible-AI frameworks &amp; **debugging agent errors**.

**Reference:** [Module 10 slides &mdash; Bias & fairness](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 10 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = "/tmp/biaa-lab-10-06"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=5):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
An agent that **acts** can act on bias at scale (deck slide 5). You can't see unfairness in an **average**
&mdash; you must measure **per group**. A common test is the **80% rule**: if the lowest group's outcome
rate is less than 80% of the highest, that's **disparate impact** worth investigating. Make bias
**visible and measured**, not assumed away. (This exact check becomes a deploy gate in Lab 12's capstone.)

In [ ]:
# records: each is {group, approved}. We measure approval rate PER group.
print("example:", {"group": "A", "approved": True})

## Build it
Complete `approval_rate_by_group` and `disparate_impact` (the 80% rule).

In [ ]:
from collections import defaultdict

def approval_rate_by_group(records):
    tot, ok = defaultdict(int), defaultdict(int)
    for r in records:
        tot[r["group"]] += 1
        if r["approved"]:
            ok[r['group']] += 1
    return {g: ok[g] / tot[g] for g in tot}

def disparate_impact(rates, threshold=0.8):
    # the 80% rule: min rate / max rate below threshold flags disparate impact
    lo, hi = min(rates.values()), max(rates.values())
    return (lo / hi) < threshold

try:
    recs = ([{"group": "A", "approved": True}] * 8 + [{"group": "A", "approved": False}] * 2 +
            [{"group": "B", "approved": True}] * 5 + [{"group": "B", "approved": False}] * 5)
    rates = approval_rate_by_group(recs)
    print("rates            :", rates)
    print("disparate impact?:", disparate_impact(rates))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- Group A approves at 0.8, group B at 0.5 &mdash; a ratio of 0.625, **below 0.8**, so disparate impact is flagged.
- The overall average (0.65) hides the gap entirely; only the **per-group** split makes it visible.
- The 80% rule doesn't prove discrimination &mdash; it flags a gap **for a human to investigate**. Measure, don't assume.

## Your turn (open task &mdash; no grader)
Add a third group C with a very low approval rate and re-run. **What good looks like:** `disparate_impact`
still fires on the worst gap across *all* groups, and you can see how an aggregate metric would have masked C
completely. This is the fairness gate the capstone uses to block an unfair batch decision.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
recs = ([{"group": "A", "approved": True}] * 8 + [{"group": "A", "approved": False}] * 2 +
        [{"group": "B", "approved": True}] * 6 + [{"group": "B", "approved": False}] * 4 +
        [{"group": "C", "approved": True}] * 2 + [{"group": "C", "approved": False}] * 8)   # C much lower
rates = approval_rate_by_group(recs)
print("rates            :", rates)                 # A 0.8, B 0.6, C 0.2
print("disparate impact?:", disparate_impact(rates))   # fires on the worst gap (C vs A), 0.25 < 0.8

---
### Key takeaway
Measure per group, not on average -- an average of 65% can hide one group at 90% and another at 40%. The 80% rule makes disparate impact visible so a human can investigate. Machines aren't neutral; measure it.

[&#8592; All Module 10 labs](./index.html) &nbsp;&middot;&nbsp; [Module 10 slides](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>